# hERG Channel Structure and Poster Drugs
**Companion notebook to CardioSafeAI-MM**

This notebook mirrors the workflow from the Day 4 Practical (AlphaFold + drug visualisation)
that we walked through for acetylcholinesterase, but applied to **the protein that
CardioSafeAI-MM is built around**: the hERG cardiac potassium channel.

| Field | Value |
|---|---|
| Gene | **KCNH2** |
| UniProt | **Q12809** |
| Protein | Voltage-gated inwardly rectifying potassium channel |
| Role | Conducts the rapid delayed-rectifier K⁺ current (Iₖᵣ) that repolarises the cardiac action potential |
| Why it matters | Off-target blockade by drugs prolongs the QT interval → arrhythmia (Torsades de Pointes). This is what every cell of `CardioSafeAI_MM.ipynb` is trying to predict. |

In Part 1 (this section) we pull the **AlphaFold-predicted structure** of hERG straight
from the EBI AlphaFold DB and inspect its per-residue confidence (pLDDT). Same recipe
the professor used for AChE — we just swap UniProt IDs.

> Later sections will add a 3D viewer and overlay our five poster drugs
> (Astemizole, Cisapride, Terfenadine, Sotalol, Verapamil) at the proposed binding cavity.

In [ ]:
# === Imports ===
# Same stack as the Day 4 practical: requests for the EBI API, Bio.PDB to parse
# the downloaded structure, py3Dmol for in-notebook 3D rendering (used later),
# matplotlib/numpy/pandas for the pLDDT plot.
import os
from pathlib import Path

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import py3Dmol
from Bio.PDB import PDBParser

# Where structure files will live. Reuses the project's data/ directory.
STRUCT_DIR = Path("data/structures")
STRUCT_DIR.mkdir(parents=True, exist_ok=True)

print("✅ imports OK")
print(f"   py3Dmol  : {py3Dmol.__version__ if hasattr(py3Dmol, '__version__') else 'installed'}")
print(f"   biopython: ", end="")
import Bio; print(Bio.__version__)

In [ ]:
# === Helper: fetch AlphaFold-predicted structure for any UniProt ID ===
# Two-step call: hit the metadata endpoint to get the pdbUrl + version, then
# download the PDB file itself. Caches to data/structures/ so repeat runs are
# instant and offline-friendly.

AF_API = "https://alphafold.ebi.ac.uk/api/prediction/{uid}"

def fetch_alphafold_pdb(uniprot_id: str, out_dir: Path = STRUCT_DIR) -> dict:
    """Download the latest AlphaFold model for `uniprot_id`.

    Returns a dict with: pdb_path, mean_plddt, version, sequence, gene.
    """
    out_dir.mkdir(parents=True, exist_ok=True)

    # 1) metadata
    r = requests.get(AF_API.format(uid=uniprot_id), timeout=30)
    r.raise_for_status()
    meta = r.json()[0]   # API returns a list; the monomer model is the first entry

    # 2) PDB file
    pdb_url = meta["pdbUrl"]
    pdb_path = out_dir / f"AF-{uniprot_id}-v{meta['latestVersion']}.pdb"
    if not pdb_path.exists():
        rp = requests.get(pdb_url, timeout=60)
        rp.raise_for_status()
        pdb_path.write_bytes(rp.content)

    return {
        "pdb_path": pdb_path,
        "mean_plddt": meta["globalMetricValue"],
        "version": meta["latestVersion"],
        "sequence": meta["sequence"],
        "gene": meta.get("gene", ""),
        "uniprot_id": uniprot_id,
        "description": meta.get("uniprotDescription", ""),
    }

In [ ]:
# === Download hERG (KCNH2 / Q12809) ===
herg = fetch_alphafold_pdb("Q12809")

print(f"Gene             : {herg['gene']}")
print(f"UniProt          : {herg['uniprot_id']}")
print(f"Description      : {herg['description']}")
print(f"AlphaFold version: v{herg['version']}")
print(f"Sequence length  : {len(herg['sequence'])} residues")
print(f"Mean pLDDT       : {herg['mean_plddt']:.2f}")
print(f"PDB file         : {herg['pdb_path']}  ({herg['pdb_path'].stat().st_size:,} bytes)")

In [ ]:
# === Per-residue pLDDT from the PDB B-factor column ===
# In AlphaFold PDB files the B-factor of every atom equals the residue's pLDDT
# (0–100). We take one value per residue (from the CA atom) and plot it along
# the sequence with the conventional 90 / 70 / 50 reference lines:
#   pLDDT > 90  → very high confidence
#   pLDDT > 70  → confident
#   pLDDT > 50  → low
#   pLDDT ≤ 50  → very low (often disordered / flexible)

parser = PDBParser(QUIET=True)
structure = parser.get_structure("hERG", str(herg["pdb_path"]))

residues, plddts = [], []
for model in structure:
    for chain in model:
        for res in chain:
            if "CA" in res:
                residues.append(res.id[1])      # residue number
                plddts.append(res["CA"].get_bfactor())

residues = np.array(residues)
plddts   = np.array(plddts)
print(f"Extracted {len(plddts)} per-residue pLDDT values "
      f"(range {plddts.min():.1f} – {plddts.max():.1f}, mean {plddts.mean():.2f})")

# Plot
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(residues, plddts, lw=0.9, color="#1f77b4")
ax.fill_between(residues, 0, plddts, alpha=0.15, color="#1f77b4")

# Confidence reference lines
for y, label, color in [
    (90, "very high (>90)", "#2ca02c"),
    (70, "confident (>70)", "#bcbd22"),
    (50, "low (>50)",       "#ff7f0e"),
]:
    ax.axhline(y, ls="--", lw=0.8, color=color, alpha=0.8)
    ax.text(residues.max() * 1.005, y, f"  {label}", color=color,
            va="center", fontsize=8)

ax.set_xlabel("Residue number")
ax.set_ylabel("pLDDT")
ax.set_title(f"AlphaFold per-residue confidence — hERG ({herg['gene']} / {herg['uniprot_id']})  "
             f"— mean = {plddts.mean():.1f}")
ax.set_ylim(0, 100)
ax.set_xlim(residues.min(), residues.max())
ax.grid(alpha=0.3)

fig.tight_layout()
out_png = Path("figures/herg_plddt.png")
out_png.parent.mkdir(exist_ok=True)
fig.savefig(out_png, dpi=140)
plt.show()
print(f"💾 saved {out_png}")

### Reading the plot — and an honest caveat

Notice the shape: a few long stretches in the *confident* / *very high* band
(the cytoplasmic PAS, cNBD and the transmembrane core S1–S6 around residues
~400–650), separated by **broad low-confidence regions** at the N-terminus,
the cytoplasmic linker, and especially the **long C-terminal tail (~residues
870–1159)** where pLDDT collapses below 50.

That's why hERG's mean pLDDT (~63) is *moderate* — substantially lower than
a compact soluble enzyme like acetylcholinesterase (typically ~90+).
**This is the honest limitation of using a predicted structure for a membrane
channel:**

- The folded core (PAS, transmembrane bundle, pore, cNBD) is reliable enough to
  reason about a binding cavity.
- The flexible / intrinsically disordered termini and linkers should **not** be
  treated as fixed geometry — they're best interpreted as "the model isn't sure
  these regions have a single conformation," which is biologically accurate for
  a channel that gates and traffics dynamically.

When we overlay our five poster drugs in the next section, we'll deliberately
focus on the **pore / inner cavity** region (the well-known Y652 / F656 binding
site), where confidence is high — and we'll be explicit that we're *not* making
claims about binding to the disordered tail.

## Part 2: Visualizing the hERG channel in 3D

The pLDDT line plot in Part 1 told us *which residues* the model is confident
about. Now we look at the actual 3D shape and overlay the same information
spatially — so the **transmembrane pore region** (the part that drugs actually
block) is visually distinct from the floppy cytoplasmic tail.

We render the cached AlphaFold PDB with **py3Dmol** (a Jupyter wrapper around
the 3Dmol.js WebGL viewer) and colour the cartoon by the per-residue **pLDDT**
confidence stored in the PDB *B-factor* column — the AlphaFold convention:

| pLDDT | Confidence | Colour |
|---|---|---|
| > 90 | very high | blue |
| 70 – 90 | confident | green / cyan |
| 50 – 70 | low | yellow |
| < 50 | very low (often disordered) | orange / red |

Reading the result: the **compact blue–green core** is the folded channel
(PAS + transmembrane bundle S1–S6 + cNBD) — that's where any drug docking
discussion is meaningful. The orange/red strands are the N- and C-terminal
regions the model couldn't pin down. No drugs yet — that's Part 3.


In [ ]:
# === Render hERG (cached PDB) as a cartoon coloured by pLDDT ===
# In AlphaFold PDBs the B-factor column stores pLDDT (0–100), so we colour the
# cartoon with a gradient over that column. `roygb` maps low→red, high→blue,
# which matches the AlphaFold convention used in the line plot above.

PDB_PATH = Path("data/structures/AF-Q12809-v6.pdb")
assert PDB_PATH.exists(), "Run Part 1 first to download the PDB."

view = py3Dmol.view(width=750, height=520)
view.addModel(PDB_PATH.read_text(), "pdb")

# Cartoon, coloured by B-factor (= pLDDT). min/max clipped to 50/90 so the
# transitions land on the standard confidence band boundaries.
view.setStyle({}, {"cartoon": {"colorscheme": {
    "prop": "b",
    "gradient": "roygb",
    "min": 50,
    "max": 90,
}}})
view.setBackgroundColor("white")
view.zoomTo()

# VS Code Jupyter renders py3Dmol via _repr_html_ on the returned view object,
# which is the most reliable path here (more so than view.show()).
# → If the cell shows blank: open the Command Palette and run
#   "Notebook: Trust", then re-run this cell. As a fallback, open the
#   notebook in a regular browser-based Jupyter instead of the VS Code panel.
view


## Part 3: The drug-binding cavity and our poster compounds

hERG blockers don't bind randomly. Decades of mutagenesis work (Mitcheson, Sanguinetti
et al., 2000) and more recent cryo-EM structures (Wang & MacKinnon, 2017) have shown
that almost all clinically relevant blockers bind in the **central pore cavity** below
the selectivity filter, gripped by two aromatic residues on the S6 helix:

- **Tyr652 (Y652)** — provides the main π/cation–π contact
- **Phe656 (F656)** — provides a complementary hydrophobic / π–π contact

Mutating either residue collapses block potency by 10–100×, which is the textbook
piece of evidence that the cavity *is* the pharmacophore.

> **Important honesty note.** What follows is **not docking.** We're highlighting the
> *known* Y652 / F656 cavity on the AlphaFold structure so we can point at where
> blockade is thought to occur, and showing our five compounds as 2D structures
> alongside. We make **no claim** about pose, affinity, or binding mode for any
> specific compound here — those would require docking (AutoDock Vina, Glide) or
> co-crystal/cryo-EM data. The blockade prediction itself comes from the chemistry
> model in `CardioSafeAI_MM.ipynb`; this section is the *structural illustration*
> that accompanies it on the poster.


In [ ]:
# === hERG cartoon (pLDDT-coloured) + Y652 / F656 cavity highlighted ===
# Re-renders the same view as Part 2 and overlays residues 652 and 656 as red
# sticks + spheres, then labels them. The PDB numbering uses the UniProt residue
# number directly (AlphaFold preserves it), so 'resi': 652 actually picks Tyr652.

PDB_PATH = Path("data/structures/AF-Q12809-v6.pdb")
assert PDB_PATH.exists(), "Run Part 1 first."

view = py3Dmol.view(width=750, height=520)
view.addModel(PDB_PATH.read_text(), "pdb")

# Base cartoon, same pLDDT colouring as Part 2
view.setStyle({}, {"cartoon": {"colorscheme": {
    "prop": "b", "gradient": "roygb", "min": 50, "max": 90,
}}})

# Highlight the binding-cavity residues
cavity = {"resi": [652, 656]}
view.addStyle(cavity, {"stick":  {"color": "red", "radius": 0.25}})
view.addStyle(cavity, {"sphere": {"color": "red", "radius": 0.45}})

# Residue labels (one per residue, at the CA position)
for resi, name in [(652, "Y652"), (656, "F656")]:
    view.addResLabels(
        {"resi": resi, "atom": "CA"},
        {"backgroundColor": "red", "fontColor": "white",
         "fontSize": 12, "showBackground": True},
    )

view.setBackgroundColor("white")
view.zoomTo(cavity)   # frame on the cavity, not the whole channel
view.zoom(0.7)        # pull back a bit so surrounding S6 is visible

# Same render path as Part 2: return view so VS Code picks up _repr_html_.
# If blank: Command Palette → "Notebook: Trust", then re-run.
view


In [ ]:
# === 2D structure grid of the five poster compounds ===
# SMILES + predicted-p + APD90 are read straight from data/worked_examples.csv
# (the same numbers the rest of the project reports on the poster), so the
# figure is guaranteed consistent with CardioSafeAI_MM.ipynb.
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Draw, AllChem

POSTER_DRUGS = ["OLEANDOMYCIN", "RIMANTADINE", "TERFENADINE", "QUINIDINE", "CISAPRIDE"]

we = pd.read_csv("data/worked_examples.csv").set_index("Drug_ID").loc[POSTER_DRUGS]

mols, captions = [], []
for name, row in we.iterrows():
    mol = Chem.MolFromSmiles(row["smiles"])
    if mol is None:
        print(f"⚠️  {name}: SMILES failed to parse — skipping")
        continue
    AllChem.Compute2DCoords(mol)
    truth   = "blocker" if row["Y"] == 1 else "non-blocker"
    caption = (f"{name.title()}  ({truth})\n"
               f"p={row['p_pred']:.3f}  |  APD90={row['APD90']:.0f} ms")
    mols.append(mol); captions.append(caption)

# returnPNG=False forces RDKit to return a real PIL.Image. Without it, RDKit
# detects the Jupyter kernel and returns an IPython.display.Image wrapper
# (PNG bytes in .data, no .save method) — which is the crash you saw.
img = Draw.MolsToGridImage(
    mols, legends=captions, molsPerRow=3,
    subImgSize=(330, 280), useSVG=False, returnPNG=False,
)

out_path = Path("figures/poster_drugs_2d.png")
out_path.parent.mkdir(exist_ok=True)
if hasattr(img, "save"):
    img.save(out_path)                       # PIL.Image — expected path
elif hasattr(img, "data"):
    out_path.write_bytes(img.data)           # IPython.Image fallback
else:
    raise TypeError(f"Unexpected MolsToGridImage return type: {type(img).__name__}")

assert out_path.exists() and out_path.stat().st_size > 0, "PNG was not written"
print(f"💾 {out_path}  ({out_path.stat().st_size:,} bytes)")

img   # PIL.Image has _repr_png_, so the grid still renders inline


### Takeaway

All five compounds are believed to act — when they act — at the **same Y652 / F656
cavity** highlighted in red above. That's the structural story.

The *quantitative* story is what `CardioSafeAI_MM.ipynb` provides:

- **Cisapride** (p ≈ 0.996), **Quinidine** (p ≈ 0.916), and **Terfenadine** (p ≈ 0.738)
  are flagged as blockers, and the O'Hara–Rudy simulation translates that block
  fraction into **action-potential prolongation** (APD90 ≈ 741, 684, 472 ms vs a
  control baseline of ~290 ms). All three were withdrawn or restricted clinically
  for QT prolongation — the model's prediction agrees with the historical record.
- **Rimantadine** (p ≈ 0.305) and **Oleandomycin** (p ≈ 0.050) come out clean, and
  their simulated APD90 stays in the ~290 ms control band — also consistent with
  the clinical record.

In one sentence: *the chemistry model decides whether the drug fits the cavity,*
*the electrophysiology model decides whether that fit matters for the heart.*
The structure shown here is the third leg — it tells the reader **where on the
protein the story is taking place**.


## Part 4: A real hERG-blocker complex (experimental cryo-EM)

Part 3 highlighted the *known* binding cavity on the AlphaFold model, but everything
about *how a drug actually sits in there* was still abstract. So now we look at a
real one.

**PDB 8ZYO — "Cryo-EM structure of astemizole-bound hERG channel"** (deposited 2024)
is exactly what it says on the tin: the hERG tetramer with the antihistamine
**astemizole** physically captured in the central pore cavity, by cryogenic
electron microscopy. The ligand atoms are *measured*, not docked, not predicted.

Astemizole itself is one of the canonical hERG blockers — it was **withdrawn
from the U.S. market in 1999** specifically because of QT prolongation and
sudden cardiac death — so this structure is the textbook example of "this is
what blockade looks like at the atomic level."

> **What this section is and isn't:**
> - ✅ A real experimental structure of hERG with a real drug bound (analogous
>   to the professor's donepezil/AChE PDB 4EY7).
> - ❌ **Not** a docking pose for any of our five poster compounds.
> - ❌ **Not** evidence that *our* predicted blockers bind exactly here — that
>   would require docking each one or a co-EM experiment.
>
> What it *does* support: the structural premise that the Y652 / F656 cavity is
> the real pharmacophore. Our three predicted blockers (Cisapride, Quinidine,
> Terfenadine) are all known clinically to act in this same pocket, so this
> image is the visual anchor for the claim made elsewhere in the project.


In [ ]:
# === Download the experimental hERG+astemizole structure (PDB 8ZYO) ===
# Cache in data/structures/ alongside the AlphaFold PDB. After download, we
# AUDIT the HETATM records and PRINT them — we only claim a bound ligand if
# the atoms are actually there. No faking.
import requests

RCSB = "https://files.rcsb.org/download/{pdb}.pdb"
TRIVIAL_HET = {"HOH", "K", "NA", "CL", "MG", "ZN", "PO4", "SO4",
               "EDO", "GOL", "PEG", "POV", "OLA", "PCW", "LMG",
               "LMN", "LMT", "CHS", "CLR", "ACE", "Y01", "CHL"}

def fetch_pdb(pdb_id, out_dir=Path("data/structures")):
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{pdb_id}.pdb"
    if not out_path.exists():
        r = requests.get(RCSB.format(pdb=pdb_id), timeout=60)
        r.raise_for_status()
        out_path.write_bytes(r.content)
    return out_path

def audit_ligands(pdb_path):
    """Return (title, {het_code: (name, atom_count)}) for non-trivial HETs."""
    title, hetnam, het_counts = "", {}, {}
    for line in pdb_path.read_text().splitlines():
        if line.startswith("TITLE") and not title:
            title = line[10:].strip()
        elif line.startswith("HETNAM"):
            code = line[11:14].strip(); name = line[15:].strip()
            hetnam[code] = (hetnam.get(code, "") + " " + name).strip()
        elif line.startswith("HETATM"):
            code = line[17:20].strip()
            if code not in TRIVIAL_HET:
                het_counts[code] = het_counts.get(code, 0) + 1
    return title, {c: (hetnam.get(c, "?"), n) for c, n in het_counts.items()}

EXP_PDB = "8ZYO"
exp_path = fetch_pdb(EXP_PDB)
title, ligands = audit_ligands(exp_path)

print(f"PDB {EXP_PDB}")
print(f"  file       : {exp_path}  ({exp_path.stat().st_size:,} bytes)")
print(f"  title      : {title}")
print(f"  ligand audit (HETATM, excluding waters / ions / lipids):")
if not ligands:
    print(f"    (none — refusing to render a 'bound' ligand that isn't there)")
else:
    for code, (name, n) in ligands.items():
        print(f"    {code}  ×{n} atoms  →  {name}")

# Hard sanity check before the render cell: there really must be a drug ligand here.
assert ligands, f"{EXP_PDB} has no non-trivial HETATM — do not claim a bound drug."
LIGAND_CODE = max(ligands, key=lambda c: ligands[c][1])   # biggest ligand by atom count
print(f"\n  → will render ligand '{LIGAND_CODE}' "
      f"({ligands[LIGAND_CODE][1]} atoms) as green sticks + surface")


In [ ]:
# === Render: grey cartoon protein + green-sticks ligand + translucent surface ===
# Same idiom as the professor's donepezil/4EY7 image: zoom on the ligand so
# the binding pocket fills the frame, surrounding helices visible as context.

view = py3Dmol.view(width=750, height=560)
view.addModel(exp_path.read_text(), "pdb")

# Protein: grey cartoon (one neutral tone, no chain colouring — the focus is the ligand)
view.setStyle({"hetflag": False},
              {"cartoon": {"color": "lightgrey", "opacity": 0.9}})

# Ligand: green sticks (thick) + translucent SAS surface
lig_sel = {"resn": LIGAND_CODE}
view.setStyle(lig_sel,
              {"stick": {"colorscheme": "greenCarbon", "radius": 0.22}})
view.addSurface(py3Dmol.SAS,
                {"opacity": 0.45, "color": "lightgreen"},
                lig_sel)

view.setBackgroundColor("white")
view.zoomTo(lig_sel)   # frame on the bound drug
view.zoom(0.85)        # pull back slightly so the surrounding S6 helices are visible

# Same render path as Parts 2/3: return view → VS Code calls _repr_html_.
# If blank: Command Palette → "Notebook: Trust", re-run.
view


### Reading this image

The green stick model in the centre is the **measured position** of astemizole
inside the hERG pore. The translucent green halo is its solvent-accessible
surface — that's the volume the drug actually occupies. The grey cartoon around
it is the channel — you're looking at four S6 helices descending into the
cavity below the selectivity filter (top of frame).

Three things this confirms structurally:

1. **The cavity is small and aromatic.** Astemizole's two aromatic rings dock
   against the Y652 / F656 side chains shown in Part 3 — exactly the
   π-stacking interaction predicted from mutagenesis (Mitcheson 2000).
2. **The drug enters from the intracellular side**, through the open channel
   gate — the textbook "trapped in the cavity" geometry that explains
   use-dependent block.
3. **The pocket is permissive about shape.** Astemizole is large, flexible,
   and amphipathic — and so are several of our predicted blockers (Cisapride,
   Terfenadine, Quinidine). The shared chemistry is *not* a precise
   pharmacophore; it's "a basic amine + at least one large aromatic group + a
   flexible linker." That's why a chemistry model trained on diverse blockers
   generalises across this scaffold space.

This is the third leg of the poster's three-leg argument:

> **Chemistry model** (p) → does the drug fit the pocket?
> **O'Hara-Rudy model** (APD90) → does that fit prolong the action potential?
> **Structure** (this image + Part 3) → and *here is the pocket* — measured.
